**Colab version of `CTD_Assignment_5`.** Same code as the Kaggle notebook; the first cell downloads the Kaggle datasets with `kagglehub` (no Kaggle login needed for public datasets). Runtime → Run all to reproduce.

Submission hub with every link for this week: https://github.com/ba-00001/brian-bazurto-python-v2/blob/assignment5/assignment5/README.md

In [ ]:
# Google Colab setup: Kaggle "Add Input" folders don't exist in Colab, so this cell
# downloads the same public Kaggle datasets with kagglehub and copies them into
# /kaggle/input. Everything below then runs exactly like it does on Kaggle.
import os
import shutil

import numpy as np
import pandas as pd

try:
    import kagglehub
except ImportError:
    %pip install -q kagglehub
    import kagglehub

DATASETS = ['chancecayaban/code-the-dream-lesson-5', 'johnmcgarvey55/code-the-dream-eclipses', 'johnmcgarvey55/code-the-dream-assignment-6']

os.makedirs('/kaggle/input', exist_ok=True)
for slug in DATASETS:
    downloaded = kagglehub.dataset_download(slug)
    target = os.path.join('/kaggle/input', slug.split('/')[1])
    shutil.copytree(downloaded, target, dirs_exist_ok=True)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Inputs and file lookup

Inputs to add with **Add Input** (search term as given in the assignment): "code the dream lesson 5" (Task 1), "Code The Dream Eclipses" (Task 2) and "Code The Dream Assignment 6" (Task 7). The helper below finds each file under `/kaggle/input` by its column names, so the exact folder name does not matter; outside Kaggle it looks in a local `data/` folder instead.

In [ ]:
import glob

INPUT_ROOT = '/kaggle/input'
LOCAL_ROOT = 'data'  # local fallback when not running on Kaggle


def find_csvs(columns=None, name=None, sep=','):
    """Return every input CSV whose path contains `name` and whose header has all of `columns`."""
    root = INPUT_ROOT if os.path.isdir(INPUT_ROOT) else LOCAL_ROOT
    found = []
    for path in sorted(glob.glob(os.path.join(root, '**', '*.csv'), recursive=True)):
        if name and name.lower() not in path.lower():
            continue
        if columns:
            try:
                header = pd.read_csv(path, sep=sep, nrows=0).columns.str.strip()
            except Exception:
                continue
            if not set(columns).issubset(header):
                continue
        found.append(path)
    if not found:
        raise FileNotFoundError(f'No input matched name={name!r} columns={columns}. Check that the Kaggle input is added.')
    return found

## Task 1 — Handling Missing Data

Load the "code the dream lesson 5" data into `df`, compare `df` with `df.dropna()`, then fill the gaps column by column and build `df2`.

In [ ]:
lesson5_path = find_csvs(columns=['Name', 'Age', 'Salary', 'Join Date', 'City'])[0]
print(lesson5_path)
df = pd.read_csv(lesson5_path)
print(df)

In [ ]:
# drop every row that has any missing value
df1 = df.dropna()
df.info()
print()
df1.info()

In [ ]:
# fill the missing values (mean for Age, median for Salary)
df['Name'] = df['Name'].fillna('Unknown')
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Salary'] = df['Salary'].fillna(df['Salary'].median())
df['Join Date'] = df['Join Date'].fillna('2020-01-01')

# only City can still be missing now
df2 = df.dropna().reset_index(drop=True)
df2['Age'] = df2['Age'].astype(int)
print(df2)

## Task 2 — Data Transformation

Load the Arkansas eclipse list (pipe separated) into `df3` and convert the `Date` column to datetime.

In [ ]:
eclipse_path = find_csvs(columns=['Date'], sep='|')[0]
print(eclipse_path)
df3 = pd.read_csv(eclipse_path, sep='|')
df3.info()
print(df3.head())

In [ ]:
# first try without errors='coerce' -- pandas stops on a date it can't handle
try:
    pd.to_datetime(df3['Date'])
except Exception as e:
    print(type(e).__name__, ':', e)

In [ ]:
# coerce turns the bad dates into NaT instead of raising
df3['Date'] = pd.to_datetime(df3['Date'], errors='coerce')
print(df3.head(20))

The dates that could not be converted are stored as `NaT` (not a time), which is the datetime version of a missing value.

## Task 3 — Validating Data Ranges

Ages outside 18 to 65 in `df2` are set to `np.nan`, then filled with the median age.

In [ ]:
df2.loc[(df2['Age'] < 18) | (df2['Age'] > 65), 'Age'] = np.nan
print(df2)

In [ ]:
df2['Age'] = df2['Age'].fillna(df2['Age'].median())
print(df2)

## Task 4 — Removing Duplicates & Outliers

Find and drop the duplicate rows in `df3`, then replace any `Age` outliers (over 100 or under 0) in `df2` with the median.

In [ ]:
df3.info()
duplicate_series = df3.duplicated()
print(df3[duplicate_series == True].head(10))
print(duplicate_series.value_counts())

In [ ]:
df3 = df3.drop_duplicates()
df3.info()

In [ ]:
# fixed thresholds for outliers; Task 3 may already have removed them
outliers = (df2['Age'] > 100) | (df2['Age'] < 0)
print('outliers found:', outliers.sum())
df2.loc[outliers, 'Age'] = df2['Age'].median()
print(df2)

## Task 5 — Standardizing Data

Clean up the `Name` text and merge the different spellings of the same city.

In [ ]:
df2['Name'] = df2['Name'].str.lower().str.strip()
print(df2)

In [ ]:
print(df2.groupby('City').agg({'Name': 'count'}))  # This will show all city names, so you can see variations

In [ ]:
city_fixes = {
    'NYC': 'New York',
    'New York City': 'New York',
    'NY': 'New York',
    'LA': 'Los Angeles',
    'L.A.': 'Los Angeles',
}
df2['City'] = df2['City'].str.strip().replace(city_fixes)
print(df2)
print(df2.groupby('City').agg({'Name': 'count'}))

## Task 6 — Encoding Categorical Variables

Label encoding with `map()` and one-hot encoding with `pd.get_dummies()` on a small color column.

In [ ]:
df_color = pd.DataFrame({'Color': ['Red', 'Blue', 'Green', 'Blue']})
df_color["Color_Label"] = df_color["Color"].map({"Red": 1, "Blue": 2, "Green": 3})
df_encoded = pd.get_dummies(df_color["Color"], prefix="Color")
print(df_color)
print(df_encoded)

## Task 7 — Consolidating Messy Files (Mini Project)

Combine the four "Code The Dream Assignment 6" files, fix spelling with fuzzy matching, fix `Zip` and `Phone` with the most common value per person, and drop duplicates.

In [ ]:
messy_paths = find_csvs(columns=['Name', 'Address', 'Zip', 'Phone'])
print(len(messy_paths), 'files')
for p in messy_paths:
    print(p)

df_all = pd.concat([pd.read_csv(p) for p in messy_paths], ignore_index=True)
df_all.info()

In [ ]:
try:
    from thefuzz import process
except ImportError:
    # Kaggle needs Internet turned on in Settings for this install
    !pip install --quiet --retries 0 --timeout 10 thefuzz
    try:
        from thefuzz import process
    except ImportError:
        # fallback when there is no internet: same extractOne idea using difflib from the standard library
        import difflib

        class process:
            @staticmethod
            def extractOne(query, choices):
                best = max(choices, key=lambda c: difflib.SequenceMatcher(None, query, c).ratio())
                return best, round(difflib.SequenceMatcher(None, query, best).ratio() * 100)
        print("thefuzz not available, using a difflib fallback")

In [ ]:
def fuzzy_fix(column):
    """Replace rare spellings in a column with the closest common spelling."""
    counts = df_all.value_counts(column)
    good_values = list(counts[counts > 2].index)
    df_all[column] = df_all[column].map(
        lambda x: x if pd.isna(x) or x in good_values else process.extractOne(x, good_values)[0]
    )

print('unique names before:', df_all['Name'].nunique(), ' addresses before:', df_all['Address'].nunique())
fuzzy_fix('Name')
fuzzy_fix('Address')
print('unique names after:', df_all['Name'].nunique(), ' addresses after:', df_all['Address'].nunique())

In [ ]:
def fix_anomaly(group):
    group_na = group.dropna()
    if group_na.empty:
        return group
    mode = group_na.mode()
    if mode.empty:
        return group
    return mode.iloc[0]

df_all["Zip"] = df_all.groupby(["Name", "Address"])["Zip"].transform(fix_anomaly)
df_all["Phone"] = df_all.groupby(["Name", "Address"])["Phone"].transform(fix_anomaly)

In [ ]:
df_all = df_all.drop_duplicates()
df_all.info()
print()
print('remaining nulls:')
print(df_all.isnull().sum())

## Task 8 — Regular Expressions for Validation

The four regex examples from the assignment, run as given.

In [ ]:
# 1. extract timestamp, level and message from log lines
log_entries = pd.Series([
    "[2023-10-26 10:00:00] INFO: User logged in",
    "[2023-10-26 10:05:30] WARNING: Invalid input",
    "[2023-10-26 10:10:15] ERROR: Database connection failed"
])
extracted_logs = log_entries.str.extract(r"\[(.*?)\]\s(\w+):\s(.*)")
print(extracted_logs)

In [ ]:
# 2. replace any bracketed placeholder with <VALUE>
text_data = pd.Series([
    "Value is {amount}.",
    "The price is [value].",
    "Cost: (number)",
    "Quantity = <qty>"
])
standardized_text = text_data.replace(
    [r"\{.*?\}", r"\[.*?\]", r"\(.*?\)", r"\<.*?\>"],
    "<VALUE>",
    regex=True,
)
print(standardized_text)

In [ ]:
# 3. keep only the columns that end in _at
df = pd.DataFrame({
    "order_id": [1, 2],
    "created_at": ["2021-01-05", "2021-01-06"],
    "updated_at": ["2021-01-07", "2021-01-08"]
})
time_cols = df.filter(regex="_at$")
print(time_cols)

In [ ]:
# 4. orders that mention shipping, any case
orders = pd.Series([
    "Order #123 has been shipped on 2021-01-05",
    "Order #124 has been cancelled",
    "Shipment #125 confirmed on 02/06/2021"
])
shipped_orders = orders[orders.str.contains("ship", case=False)]
print(shipped_orders)

## Task 9 — Reflection & Validation

**Most common data issues found**
- Missing values in the lesson 5 data (Name, Age and City had gaps; Salary and Join Date did not).
- Values outside a sensible range, like ages of -1, 104 and 120.
- Dates pandas could not parse in the eclipse file, which turned into `NaT` once I used `errors='coerce'`.
- Exact duplicate rows in the eclipse list.
- Inconsistent text: extra spaces and mixed case in names, and the same city written as "NYC" and "New York" or "LA" and "Los Angeles".
- In the four messy files, misspelled names and addresses plus conflicting or missing zip codes and phone numbers for the same person.

**Which techniques worked best**
- `fillna()` with a strategy per column (a placeholder for text, mean or median for numbers). Median was the safer choice for salary because a few large values pull the mean up.
- Boolean masks with `.loc` for range checks and outliers, since they read like the rule itself.
- `pd.to_datetime(..., errors='coerce')` to keep going and then deal with the bad dates afterwards.
- `duplicated()` / `drop_duplicates()` to find and remove repeats.
- `str.lower().str.strip()` plus a replacement dictionary for text cleanup.
- For the mini project, fuzzy matching (`thefuzz`) for typos and a group mode (`groupby().transform()`) for zip and phone. Together they cut about 1600 rows down to close to 400 real records.

**How I could automate cleaning in a real workflow**
- Put each step in a function (fill, validate ranges, standardize text, dedupe) and run them in the same order on every new file.
- Keep the rules in one place, such as the valid age range or the city mapping dictionary, so they are easy to update.
- Add checks that run after cleaning, for example asserting there are no nulls in required columns and no ages out of range, and log how many rows each step changed.
- Schedule the script to run whenever new data lands, and save the cleaned output to a file or database for the next step.